# Creating Designs by Leveraging Stable Diffusion and Gradio UI

## Overview

In this project, you will explore the transformative potential of Hugging Face's Stable Diffusion model and Gradio UI in the creative design industry. The project outlines hypothetical scenarios to demonstrate AI's ability to generate images from text prompts. These scenarios offer a glimpse into a future where technology enhances creativity.

## Instructions

- Review the learning materials and the Gradio documentation provided for the project.
- Read the sections on situation, task, action, and result carefully to understand the assignment.
- Complete and submit your assignment as a Python module (.py file) through the Learning Management System (LMS).
- Adhere closely to the provided guidelines, ensuring your submission contains all necessary code and is runnable from the command line.

## Situation

You work as a designer at a cutting-edge creative agency. Your task involves revolutionizing visual content creation for digital marketing campaigns, especially for high-profile clients such as Netflix. You have access to an innovative web-based platform. This platform combines Hugging Face's Stable Diffusion model with Gradio's web interface.

## Task

Your task is to create a platform that generates bespoke images and designs for banners and posters by entering text prompts. By leveraging the capabilities of Stable Diffusion (`CompVis/stable-diffusion-v1-4` is a good model to start with), you aim to transform abstract ideas into tangible, visually compelling designs. These designs will captivate audiences and enhance promotional efforts for Netflix's latest campaigns.

## Action

- Import essential libraries such as Gradio for building web interfaces for machine learning models, the Hugging Face Diffusers library for loading Stable Diffusion, and PyTorch for model execution. These libraries collectively enable the transformation of text prompts into images.
- Create a function named `generate_image` to oversee the image generation process using text prompts.
- Set up the Gradio interface using `Interface`, including a textbox for text prompt input and an image display for showcasing generated images.
- Launch the Gradio interface to make it available for user interaction.

## Result

Upon completing this project, you will submit a Python module (.py file). This file will showcase the innovative potential of integrating Hugging Face's Stable Diffusion model and Gradio UI in the creative design industry. It focuses on generating customized visual content for Netflix campaigns. The module should be runnable from the command line (e.g., `python image_generator.py`) and launch the Gradio interface automatically.

**Gradio Documentation:** [Document](https://drive.google.com/file/d/1Fpvq-VM5N5mM3g3GnaAZkTpoUHb2jZq_/view?usp=drive_link)

In [ ]:
%pip install --quiet --upgrade pip
%pip install --quiet gradio diffusers[torch] triton-windows
%pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cu130

In [ ]:
# imports
import gc
import gradio as gr
import torch
from diffusers import DiffusionPipeline, StableDiffusion3Pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
gc.collect()
torch.cuda.empty_cache()

def generate_image (prompt, negative_prompt, guidance_scale, seed):
    # Take model and generate an image from prompt and seed
    # model_id = "stabilityai/stable-diffusion-xl-base-1.0"
    model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
    pipe = DiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16, use_safetensors=True, variant="fp16")
    pipe = pipe.to(device)
    pipe.safety_checker = lambda images, **kwargs: (images, [False] * len(images))
    # refiner = DiffusionPipeline.from_pretrained(
    # "stabilityai/stable-diffusion-xl-refiner-1.0",
    #     text_encoder_2=pipe.text_encoder_2,
    #     vae=pipe.vae,
    #     torch_dtype=torch.float16,
    #     use_safetensors=True,
    #     variant="fp16",
    #     )
    # refiner = refiner.to(device)
    generator = torch.Generator(device).manual_seed(seed)
    pipe.to(device)
    pipe.enable_model_cpu_offload()
    image = pipe(prompt, negative_prompt=negative_prompt, generator=generator, height=328, width=720, guidance_scale=guidance_scale).images
    # image1 = refiner(prompt, image=image).images[0]
    gc.collect()
    torch.cuda.empty_cache()
    
    # If torch.compile was used:
    torch._dynamo.reset()
    torch.cuda.synchronize()
    return image[0]


In [ ]:
def_prompt = "Create a banner ad for 'Gears Ltd.' They make gears. Use blue for background color, and steel for the gears"
demo = gr.Interface(fn=generate_image , inputs=[gr.Textbox(label="Prompt", value=def_prompt),
                                             gr.Textbox(label="Negative Prompt", value="text, words, logo, watermarks, signatures"),
                                             gr.Slider(0, 20, step=0.1, label="Guidance Scale", value=7.5),
                                             gr.Number(label="Seed", value=691)],
                                     outputs=[gr.Image(label="Banner Background")],)

demo.launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


Loading weights: 100%|██████████| 196/196 [00:00<00:00, 290.33it/s, Materializing param=text_model.final_layer_norm.weight]
CLIPTextModel LOAD REPORT from: C:\Users\Rick\.cache\huggingface\hub\models--stable-diffusion-v1-5--stable-diffusion-v1-5\snapshots\451f4fe16113bff5a5d2269ed5ad43b0592e9a14\text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 396/396 [00:01<00:00, 264.96it/s, Materializing param=visual_projection.weight]
StableDiffusionSafetyChecker LOAD REPORT from: C:\Users\Rick\.cache\huggingface\hub\models--stable-diffusion-v1-5--stable-diffusion-v1-5\snapshots\451f4fe16113bff5a5d2269ed5ad43b0592e9a14\safety_checker
Key                                               | Status     |  | 
---------------------